Batarya Ömür Tahmini Projesi (Battery Aging Prediction Project)

Kaydedilmiş verileri alıp model için kulanılacak matrisleri oluşturma
(Getting the saved data and building the matrices to use for model)

1. Gerekli kütüphaneleri koda ekledim (Added essential libraries to the code)

In [1]:
#Got the essential libraries
import h5py
import numpy as np
import gc
import scipy
from scipy.signal import savgol_coeffs,savgol_filter
from numpy.lib.stride_tricks import sliding_window_view

2. Yumuşatma için veri sızıntısını engelleyen kaydırmalı pencere kullanarak savitzky-golay filtresi tanımladım
(Defined Savitzky-Golay filter for smoothing with sliding window that prevents data leakage).

In [2]:
#Defined Savitzky-Golay smoothing filter with window sliding and preventing data leakage by only looking past values
def savgol_smoothing (data, window_l, poly, method):
    #Defined coefficients
    coeffs = savgol_coeffs(window_l, poly, pos=window_l-1)
    #Picked the method based on input and smoothed the data
    if method == "correlate":
        smoothened_data = np.correlate(data, coeffs, "valid")
    elif method == "view":
        windows = sliding_window_view(data, window_l)
        smoothened_data = windows @ coeffs
    else:
        raise ValueError("Wrong method! Pick 'correlate' or 'view'")
    #Output the data
    return smoothened_data

3. Verilerin kaydedildiği h5 dosyasını okuyup, içindeki veri başlıklarını inceleme
(Read the h5 files that data saved,inspecting the key data sets).

In [3]:
#Read the battery data from h5 file
with h5py.File("FastCharge_Interpolated_Data.h5", "r") as h5f:
    battery_names = list(h5f.keys())
    print(f"Total: {len(battery_names)} batteries")

    #Examined the first battery data
    first_battery = battery_names[0]
    group = h5f[first_battery]
    
    print(f"\nBattery: {first_battery}")
    print("Data Sets(Keys):", list(group.keys()))

    #Took a look at the size of the full battery data and discharge part of battery data
    a = len(group["discharge_capacity"])//2
    print(a*2)
    c = {k:v[1:a-1] for k,v in group.items()}
    print(len(c["discharge_capacity"]))

    #Checked the names of all batteries
    for name,battery in h5f.items():
        print(name)

Total: 139 batteries

Battery: FastCharge_000000_CH19_structure.4_4c_55per_6c
Data Sets(Keys): ['charge_capacity', 'current', 'cycle_index', 'discharge_capacity', 'internal_resistance', 'temperature', 'voltage']
982
489
FastCharge_000000_CH19_structure.4_4c_55per_6c
FastCharge_000001_CH16_structure.3_7c_31per_5_9c_newstructure
FastCharge_000001_CH30_structure.3_7c_31per_5_9c_newstructure
FastCharge_000001_CH38_structure.3_7c_31per_5_9c_newstructure
FastCharge_000002_CH10_structure.5c_67per_4c_newstructure
FastCharge_000002_CH18_structure.5c_67per_4c_newstructure
FastCharge_000002_CH2_structure.5c_67per_4c_newstructure
FastCharge_000002_CH34_structure.5c_67per_4c_newstructure
FastCharge_000002_CH42_structure.5c_67per_4c_newstructure
FastCharge_000002_CH47_structure.5c_67per_4c_newstructure
FastCharge_000002_CH7_structure.5c_67per_4c_newstructure
FastCharge_000003_CH39_structure.7c-30per_3_6c
FastCharge_000003_CH40_structure.7c-30per_3_6c
FastCharge_000004_CH1_structure.3_6c-80per_3_6c
F

4. h5 dosyasından bize gereken verileri alıp, temizleme ve yumuşatma yaptıktan sonra, bunlar ile yeni bir tane h5py dosyası oluşturdum
(Took the essential data from h5 file, cleaned and smoothened,then built a new h5py file with them).

In [ ]:
#Defined a nominal capacity value based on the information about the batteries(information link: https://data.matr.io/1/projects/5c48dd2bc625d700019f3204)
nominal_capacity = 1.1

#Opened the h5 file with unprocessed data 
with h5py.File("FastCharge_Interpolated_Data.h5", "r") as h5f:

    #Created a new h5 file to save processed data
    with h5py.File("FastCharge__Features_Target.h5", "w") as h5f_r:

        #Got the values to process the data from old h5 file
        for name,battery in h5f.items():
            half = len(battery["discharge_capacity"]) // 2
            discharge = {k:v[1:half-1] for k,v in battery.items()}
            capacity = discharge["discharge_capacity"]
            voltage = discharge["voltage"]
            ir = discharge["internal_resistance"]
            temperature = discharge["temperature"]

            #Smoothened Capacity values using centered Savitzky-Golay filter(with a NaN or Inf protection)
            smooth_capacity = np.zeros((half-2,1000))
            for i in range(half-2):
                #Grabbed the capacity values
                cap_row = capacity[i, :].copy()
                
                #Transformed Inf values to NaN and unmasked the NaN values
                cap_row = np.where(np.isinf(cap_row), np.nan, cap_row)
                mask = np.isnan(cap_row)
                
                #Fixed the NaN if exists
                if mask.any():
                    valid_indices = np.flatnonzero(~mask) 
                    
                    #Used interpolation with valid values
                    if len(valid_indices) > 0:
                        cap_row[mask] = np.interp(np.flatnonzero(mask), valid_indices, cap_row[~mask])
                    else:
                        #Replaced the cycle with last cycle if its completely abnormal
                        if i > 0:
                            cap_row = smooth_capacity[i-1, :].copy()
                        else:
                            cap_row = np.zeros(1000)
                
                #Saved the smoothened and possibly fixed values
                smooth_capacity[i,:] = savgol_filter(cap_row, window_length=21, polyorder=2)

            #Defined features
            max_capacity = np.zeros(half-2)
            last_ir = np.zeros(half-2)
            temp_var = np.zeros(half-2)
            smooth_Q_V_derivative_kurtosis = np.zeros(half-2)
            smooth_Q_V_derivative_skewness = np.zeros(half-2)
            smooth_Q_V_derivative_peakheight = np.zeros(half-2)
            smooth_Q_V_derivative_variance = np.zeros(half-2)

            #Calculated the dQ/dV values with smoothened capacity values
            smooth_Q_V_derivative_f = np.gradient(smooth_capacity,voltage[0],axis=1)

            #Calculated the features for each cycle
            for i in range(half-2):
                max_capacity[i] = np.max(smooth_capacity[i,:])
                last_ir[i] = ir[i,-1]
                temp_var[i] = np.var(temperature[i,:])
                smooth_Q_V_derivative_kurtosis[i] = scipy.stats.kurtosis(smooth_Q_V_derivative_f[i,:])
                smooth_Q_V_derivative_skewness[i] = scipy.stats.skew(smooth_Q_V_derivative_f[i,:])
                smooth_Q_V_derivative_peakheight[i] = np.max(abs(smooth_Q_V_derivative_f[i,:]))
                smooth_Q_V_derivative_variance[i] = np.log(np.var(smooth_Q_V_derivative_f[i,:]) + (1e-8))

            #Smoothened the features
            smooth_max_capacity = savgol_smoothing(max_capacity, window_l=91 ,poly=2, method="correlate")
            smooth_last_ir = savgol_smoothing(last_ir, window_l=91 ,poly=2, method="correlate")
            smooth_temp_var = savgol_smoothing(temp_var, window_l=91,poly=1,method="correlate")
            smoother_QV_derivative_kurtosis = savgol_smoothing(smooth_Q_V_derivative_kurtosis, window_l=91 ,poly=2, method="correlate")
            smoother_QV_derivative_skewness = savgol_smoothing(smooth_Q_V_derivative_skewness, window_l=91 ,poly=2, method="correlate")
            smoother_QV_derivative_peakheight = savgol_smoothing(smooth_Q_V_derivative_peakheight, window_l=91 ,poly=2, method="correlate")
            smoother_QV_derivative_variance_log = savgol_smoothing(smooth_Q_V_derivative_variance, window_l=91 ,poly=2, method="correlate")

            #Defined SOH values by normalizing max capacity values from each cycle
            initial_capacity = smooth_max_capacity[0] #Left the initial capacity here, it can be used to calculate SOH as well
            soh_capacity = smooth_max_capacity/nominal_capacity

            #Built the features matrice for the selected battery
            battery_features = np.column_stack((smooth_last_ir,smooth_temp_var,smoother_QV_derivative_kurtosis,smoother_QV_derivative_skewness,
                                                smoother_QV_derivative_peakheight,smoother_QV_derivative_variance_log))
            
            battery_feature_names = ["Internal_Resistance","Temperature_Variance","dQ/dV_Kurtosis","dQ/dV_Skewness","dQ/dV_Peakheight","dQ/dV_Variance_log"]
            battery_target_name = ["State_of_Health_Capacity"]
            battery_group = h5f_r.create_group(name)

            #Saved the features in the new h5 file
            battery_group.create_dataset(
            name="Features", 
            data=battery_features, 
            compression="gzip",      
            compression_opts=4       
            )

            
            #Saved the target in the new h5 file
            battery_group.create_dataset(
            name="Target", 
            data=soh_capacity, 
            compression="gzip",      
            compression_opts=4       
            )
            
            #Saving the feature names and the target name in h5 file
            battery_group["Features"].attrs["Names"]=battery_feature_names
            battery_group["Target"].attrs["Name"]=battery_target_name
            
            
            #Deleted the leftovers for better performance
            del battery_features
            del smooth_max_capacity
            
gc.collect()

10